# Random Forests — Bagging, OOB Intuition, and Feature Importance

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb12_random_forests_importance.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how **bootstrap aggregation (bagging)** plus **random feature subsets** turn high-variance single trees into stable ensembles, on both classification and regression spines.
2. Fit `RandomForestClassifier` and `RandomForestRegressor` and demonstrate their CV-score lift over the single tree from nb11 and the **Week-2 reference** baselines.
3. Tune `n_estimators` and `max_features` under the same one-standard-error rule used in nb11.
4. Read the **Out-of-Bag (OOB) score** as a free, non-redundant validation signal — and know when to trust it vs cross-validation.
5. Build the **four-method feature-importance reconciliation table** (linear-coefficient / impurity / permutation / drop-column) — the course-wide reference table that nb15 relies on for interpretation.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification track (RF tuning) and Exercise 2 on the regression track (RF tuning). Complete both before submitting your notebook.

---

## 💼 Why This Matters

The single decision tree from nb11 is high-variance: a small change in the training data flips the root split, and from there the entire tree restructures. Both the State Health Department's review board and HomeValue Analytics' deployment council asked the same follow-up question after seeing the tree: *"What happens to this model if you retrain it on next month's data?"*

The honest answer is *"a different tree"*, which is not the answer either stakeholder wanted. **Random forests** fix this by training many trees on bootstrap samples with random feature subsets, then averaging their predictions. No single tree dominates; the forest votes. Variance drops sharply, predictions stabilize, and you keep the ability to handle non-linearity that linear baselines cannot.

The cost is a loss of single-path interpretability — you can no longer trace a flowchart from root to leaf because there are 100 of them. In exchange you get:

- **Stable predictions** under retraining (bias unchanged, variance roughly cut by `1/n_estimators`).
- **Built-in OOB validation** — every tree was trained on a bootstrap sample, so the ~37% of training data left out of each tree is a free held-out set.
- **Four different views of feature importance** that, when they disagree, tell you something interesting about the data.

By the end of today, both the State Health Department's screening forest and HomeValue's price-prediction forest should beat their respective **Week-2 references** by a CI-clear margin. If they do not, the random forest has not earned its added complexity, and the linear baseline ships. That CI-clear discipline is the bar nb14's selection ceremony will enforce on a five-candidate field per spine.

A question that often comes up here is *"if a forest is just an average of trees, why do I need the four-method importance table?"* Because the average of trees is opaque. You cannot read 100 flowcharts. The importance table is the diagnostic that lets you answer the **"what is the model paying attention to?"** question without re-deriving it from the trees. Section 6 makes that table; nb15 lifts it into the M3 milestone.

---

## 1. Setup — Imports, References, Helpers

The setup cell does five things at once: imports the random-forest estimators for both spines, locks `RANDOM_SEED = 474`, defines the **Week-2 reference pipelines** (`reference_clf` = LogReg(C=1.0); `reference_reg` = OLS) carried over from nb09, and registers the plot helpers nb11 introduced (`plot_train_val_curve`, `plot_predicted_vs_actual`) plus three new ones for this notebook (`plot_cv_ci`, `plot_importance_bars`, `plot_importance_heatmap`).

> 💡 **Gemini Prompt:** "Set up imports for sklearn RandomForestClassifier, RandomForestRegressor, DecisionTreeClassifier, DecisionTreeRegressor, LogisticRegression, LinearRegression, permutation_importance, cross_val_score, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing, StandardScaler, Pipeline. Set RANDOM_SEED = 474. Define reference_clf and reference_reg as the Week-2 baseline pipelines. Define helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci (dot plot with 95% CI bars across N models), plot_importance_bars (horizontal bar chart with optional error bars), plot_importance_heatmap (heatmap of feature ranks across methods)."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474`, `reference_clf` and `reference_reg` defined
> - [ ] All five plot helpers callable
> - [ ] No import errors


In [ ]:
# Setup — imports, seed, Week-2 references, plot helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# --- Course color convention ---
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'

# --- Week-2 reference models (from nb09's CI-overlap test) ---
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

# --- Helper 1 (from nb11): train-vs-CV overfitting curve ---
def plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax,
                         color_train=GREY, color_val=CLF_COLOR):
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train', linewidth=2, color=color_train)
    ax.errorbar(xs, val_mean, yerr=val_std, marker='s', label='5-fold CV ± SD',
                linewidth=2, capsize=5, color=color_val)
    ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

# --- Helper 2 (from nb11): predicted-vs-actual scatter ---
def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual',
                             color=REG_COLOR):
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo, hi = float(min(np.min(y_true), np.min(y_pred))), float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

# --- Helper 3 (NEW): CV-CI dot plot (with 95% CI from Student's t) ---
def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5):
    """scores_dict: {model_name: array of fold scores}; draws 95% CI from t-dist."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    ax.errorbar(df['mean'], df['name'], xerr=df['half_w'],
                fmt='o', capsize=6, linewidth=2, color=color, markersize=10)
    for _, r in df.iterrows():
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                r['name'], f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Helper 4 (NEW): horizontal importance bar chart with optional error bars ---
def plot_importance_bars(importances, names, ax, errors=None, color=CLF_COLOR,
                         title='Feature importance', top_n=15):
    """Horizontal bar chart, sorted descending. errors optional (e.g., permutation SD)."""
    df = pd.DataFrame({'name': names, 'imp': importances})
    if errors is not None: df['err'] = errors
    df = df.sort_values('imp', ascending=True).tail(top_n)
    if errors is not None:
        ax.barh(df['name'], df['imp'], xerr=df['err'], color=color, edgecolor='black', capsize=4)
    else:
        ax.barh(df['name'], df['imp'], color=color, edgecolor='black')
    ax.set_xlabel('Importance')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Helper 5 (NEW): rank-disagreement heatmap (4-method reconciliation) ---
def plot_importance_heatmap(rank_df, ax, title='Feature rank across methods', top_n=15):
    """rank_df: rows = features, columns = methods; values = rank (1 = most important)."""
    # Show only top_n features by best (smallest) rank across methods
    best_rank = rank_df.min(axis=1)
    keep = best_rank.nsmallest(top_n).index
    sub = rank_df.loc[keep].sort_values(rank_df.columns[0])
    im = ax.imshow(sub.values, cmap='RdYlGn_r', aspect='auto', vmin=1, vmax=rank_df.shape[0])
    ax.set_xticks(range(sub.shape[1])); ax.set_xticklabels(sub.columns, rotation=20, ha='right')
    ax.set_yticks(range(sub.shape[0])); ax.set_yticklabels(sub.index)
    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            ax.text(j, i, int(sub.values[i, j]), ha='center', va='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Rank (1 = most important)', shrink=0.7)

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print(f"✓ Week-2 references: reference_clf, reference_reg")
print(f"✓ Helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci, "
      f"plot_importance_bars, plot_importance_heatmap")


**Reading the output:**

Five plot helpers are now in scope. Two carry over from nb11 (`plot_train_val_curve`, `plot_predicted_vs_actual`). Three are new for this notebook: `plot_cv_ci` produces the dot-plot-with-error-bars that you will see in Section 8's comprehensive comparison and again in nb14's selection ceremony; `plot_importance_bars` standardizes the horizontal feature-importance chart used in Sections 6 and 7; `plot_importance_heatmap` renders the four-method reconciliation table — the visualization that turns rank disagreement across methods into the central pedagogical payload of Section 6.

The two reference pipelines `reference_clf` and `reference_reg` are exactly what survived nb09's CI-overlap test on the two datasets. They will appear in Section 8 as the lower bars on the comprehensive-comparison plot — the floor every Random Forest variant has to clear by a CI-clear margin.

**Key takeaway:** All five helpers + both Week-2 references are defined once, reused across every section. Helper definitions never change between notebooks; their signatures stay stable so muscle memory transfers when you reuse them in your own M3 work.

---

## 2. From Single Tree to Forest — The Bagging Idea

A bootstrap sample is a draw of `n` samples **with replacement** from the original `n`-sample training set. On average about 63% of the original samples appear in any given bootstrap (some appear twice or more, others not at all). The remaining ~37% are **out-of-bag** for that bootstrap — they form a free held-out set without spending any of the test data.

Bagging fits one base learner on each bootstrap. For trees, the variance reduction has a tidy upper bound: averaging `B` independent trees reduces variance by a factor of `1/B`. Trees fit on different bootstraps are not perfectly independent (they share much of the underlying training data), so the realized reduction is smaller than `1/B` — but still substantial, and Section 4 will quantify it.

The plot below makes the idea concrete: a single feature (`mean radius` from breast cancer) drawn under five separate bootstrap samples. Each KDE is slightly different — that variability across bootstraps is exactly what averaging a forest cancels out.

In [ ]:
# Bootstrap KDEs — visualize what each base learner actually sees
data_clf = load_breast_cancer(as_frame=True)
X_show = data_clf.data['mean radius'].values
rng = np.random.default_rng(RANDOM_SEED)

fig, ax = plt.subplots(figsize=(11, 5))
xs = np.linspace(X_show.min(), X_show.max(), 400)
for b in range(5):
    boot = rng.choice(X_show, size=len(X_show), replace=True)
    # Simple Gaussian KDE
    bw = 1.06 * boot.std() * len(boot) ** (-1/5)
    kde = np.exp(-0.5 * ((xs[:, None] - boot[None, :]) / bw) ** 2).sum(axis=1) / (len(boot) * bw * np.sqrt(2*np.pi))
    ax.plot(xs, kde, alpha=0.7, linewidth=2, label=f'Bootstrap {b+1}')

# Overlay original distribution as reference
bw0 = 1.06 * X_show.std() * len(X_show) ** (-1/5)
kde0 = np.exp(-0.5 * ((xs[:, None] - X_show[None, :]) / bw0) ** 2).sum(axis=1) / (len(X_show) * bw0 * np.sqrt(2*np.pi))
ax.plot(xs, kde0, color='black', linewidth=3, linestyle='--', label='Original training data', alpha=0.8)

ax.set_xlabel('mean radius')
ax.set_ylabel('density')
ax.set_title('Five bootstrap samples of the training data — each base learner sees a slightly different slice',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Each bootstrap is similar but not identical — a different tree fits a different shape.")
print("💡 Random forests average across this variability; the forest's variance is far below any single tree's.")


**Reading the output:**

All five bootstrap KDEs cluster around the original training distribution but diverge in details — different peaks, different tails, different local densities. A decision tree fit on each bootstrap will pick slightly different split thresholds in `mean radius` and slightly different child structures, even though the underlying data is the same. The variance across these five trees would be the variance you saw in nb11's `max_depth=20` overfit row — large, fold-dependent, fragile under retraining.

The forest's prediction is the average (for regression) or majority vote (for classification) over all these slightly different trees. Two trees that disagree on a borderline patient cancel each other out at the vote; two trees that both vote malignant on an obvious case reinforce each other. The averaging mechanism turns the ensemble into a much smoother, much more stable predictor than any single tree.

A question that often comes up here is *"if bootstrap is just sampling with replacement, why does it work?"* Two reasons. First, each tree's training set is genuinely different — fit one tree on the same data twice and you get the same tree, but fit it on two different bootstraps and you get two different trees. Second, the trees' errors are partially uncorrelated — when tree A is wrong on a sample, tree B is often right, and the average is closer to truth than either individual prediction. The math behind this is the classic variance-reduction-by-averaging result: if `B` independent estimators each have variance `σ²`, their average has variance `σ²/B`. Bagged trees are not fully independent (they share data), so the realized factor is between `1/B` and `1`, but the direction is always correct.

**Key takeaway:** The bootstrap creates the diversity; the averaging cashes the diversity in as variance reduction. Section 3 measures the cash-in directly — single tree's CV variability vs forest's CV variability under identical conditions.

---

## 3. Load Both Datasets — Two Locked Test Sets, Two CV Splitters

Same locking discipline as nb11. Both spines split 70/30 with `random_state=RANDOM_SEED`; test sets stay sealed until nb14's ceremony. Classification uses stratified CV; regression uses plain `KFold`. The variable suffixes `_clf` / `_reg` keep the namespaces unconfusable.

In [ ]:
# --- Classification track: Wisconsin breast cancer ---
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Regression track: California Housing ---
data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("=== CLASSIFICATION (Wisconsin Breast Cancer) ===")
print(f"  Train: {len(X_train_clf):>6} | Test: {len(X_test_clf):>6} (LOCKED until nb14)")
print()
print("=== REGRESSION (California Housing) ===")
print(f"  Train: {len(X_train_reg):>6} | Test: {len(X_test_reg):>6} (LOCKED until nb14)")


**Reading the output:**

Identical to nb11's split — the same `random_state=RANDOM_SEED` produces the same partition, so the CV scores you compute here are directly comparable to nb11's single-tree numbers. That is the whole point of locking the seed: when nb14's ceremony declares a champion, the chain of CV evidence from nb11 → nb12 → nb13 → nb14 is on identical splits.

**Key takeaway:** Same splits as nb11 → results are directly comparable. From here, every model's CV mean can be quoted alongside nb11's tree numbers without an asterisk.

---

## 4. Single Tree vs Random Forest — Paired

The headline comparison: a single `DecisionTree(depth=5)` versus a `RandomForest(100 trees, depth=5)` on each spine. Same depth, same data, same CV folds. The forest should beat the tree on **both** mean score (modestly) and CV standard deviation (substantially) — the latter is the variance-reduction payoff making itself visible.

> 💡 **Gemini Prompt:** "Fit DecisionTreeClassifier(depth=5) and RandomForestClassifier(n_estimators=100, depth=5, n_jobs=-1) on X_train_clf, evaluate via 5-fold CV ROC-AUC. Same for DecisionTreeRegressor(depth=5) vs RandomForestRegressor(n_estimators=100, depth=5, n_jobs=-1) on X_train_reg with R². Report mean and SD for both spines; build a 1×2 CV-CI dot plot using the plot_cv_ci helper."
>
> **After running, verify:**
> - [ ] Forest CV mean > tree CV mean on both spines
> - [ ] Forest CV SD < tree CV SD on both spines (the variance-reduction signal)
> - [ ] Both panels use the plot_cv_ci helper (95% CI from Student's t)


In [ ]:
# Single tree vs forest — paired comparison
tree_clf  = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED)
forest_clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1)
tree_reg  = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)
forest_reg = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1)

clf_scores = {
    'Single Tree (depth=5)': cross_val_score(tree_clf,   X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (100×5)': cross_val_score(forest_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_scores = {
    'Single Tree (depth=5)': cross_val_score(tree_reg,   X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Random Forest (100×5)': cross_val_score(forest_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
}

print("=== CLASSIFICATION (5-fold CV ROC-AUC) ===")
for name, s in clf_scores.items():
    print(f"  {name}:  mean = {s.mean():.4f}  SD = {s.std(ddof=1):.4f}")
print()
print("=== REGRESSION (5-fold CV R²) ===")
for name, s in reg_scores.items():
    print(f"  {name}:  mean = {s.mean():.4f}  SD = {s.std(ddof=1):.4f}")

# Side-by-side CV-CI dot plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_cv_ci(clf_scores, 'ROC-AUC', 'Classification — single tree vs forest', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_scores, 'R²',      'Regression — single tree vs forest',     axes[1], color=REG_COLOR)
fig.suptitle('Variance reduction in action — forest CIs are tighter than the single tree on both spines',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On both spines the random forest beats the single tree on **both** mean and SD. The mean lift is usually modest (1–3 score points) — single trees are not catastrophically bad on these datasets — but the SD reduction is dramatic. The forest's 95% CI is typically half the width of the single tree's, which is exactly the variance-reduction-by-averaging mechanic in action.

A question that often comes up here is *"if 100 trees average to a slightly better mean than one tree, why not 1000 trees?"* Two reasons. First, the marginal CV gain from each additional tree shrinks fast — Section 4 will show the curve plateauing around 100–200 trees on both spines. Second, the variance-reduction math has a hard floor: when bootstrap samples are highly correlated (which they are, by construction), the realized reduction is less than `1/B` and asymptotes to a non-zero residual. Past that asymptote, more trees just buy you fit time, not predictive lift. Section 5 covers `max_features`, which is the lever that controls how correlated the trees are — and which therefore controls how low the asymptote sits.

**Key takeaway:** Forests beat trees on both measures, but the bigger win is the variance reduction (tighter CI) — not the mean lift. That stability is what makes the forest defensible to stakeholders who will retrain monthly.

---

## 5. Tuning Random Forests — `n_estimators` and `max_features`

Random forests have two key hyperparameters beyond what trees have: **`n_estimators`** (how many trees in the ensemble) and **`max_features`** (how many features each split considers). Both shape the bias/variance trade-off.

- More `n_estimators` → lower variance, eventual plateau, no risk of overfitting per se (more trees just smooth the average).
- Smaller `max_features` → more diverse trees → lower correlation → larger variance reduction at the cost of some single-tree quality. The sklearn defaults are `sqrt(n_features)` for classification and `1.0` (all features) for regression — the latter is often suboptimal and worth tuning.

Two paired sweeps: one for `n_estimators`, one for `max_features`. Both use 3-fold CV here (instead of 5) to keep the regression-grid runtime manageable on Colab; the headline plots in later sections will use full 5-fold CV.

> 💡 **Gemini Prompt:** "Sweep n_estimators in [10, 25, 50, 100, 200] for both spines using 3-fold CV (scoring='roc_auc' for clf, 'r2' for reg). Then sweep max_features in ['sqrt', 'log2', 0.3, 0.5, None] for both spines at n_estimators=100. Two paired figures using plot_train_val_curve for the n_estimators sweep, plus a side-by-side bar plot for max_features."
>
> **After running, verify:**
> - [ ] n_estimators curves both plateau (clf around 50–100, reg around 100)
> - [ ] max_features sweep shows that 'sqrt'/0.3 typically beat 'None' on regression
> - [ ] All cells use n_jobs=-1


In [ ]:
# Use 3-fold CV for these inline sweeps to keep regression runtime modest on Colab
cv_clf_3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
cv_reg_3 = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

# --- Sweep 1: n_estimators ---
n_est_grid = [10, 25, 50, 100, 200]

clf_train, clf_val_mean, clf_val_std = [], [], []
for n in n_est_grid:
    m = RandomForestClassifier(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train_clf, y_train_clf)
    clf_train.append(m.score(X_train_clf, y_train_clf))
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='roc_auc', n_jobs=-1)
    clf_val_mean.append(s.mean()); clf_val_std.append(s.std(ddof=1))

reg_train, reg_val_mean, reg_val_std = [], [], []
for n in n_est_grid:
    m = RandomForestRegressor(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train_reg, y_train_reg)
    reg_train.append(m.score(X_train_reg, y_train_reg))
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    reg_val_mean.append(s.mean()); reg_val_std.append(s.std(ddof=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_train_val_curve(n_est_grid, clf_train, clf_val_mean, clf_val_std,
                     'n_estimators', 'Accuracy (train) / ROC-AUC (CV)',
                     'Classification', axes[0], color_val=CLF_COLOR)
plot_train_val_curve(n_est_grid, reg_train, reg_val_mean, reg_val_std,
                     'n_estimators', 'R² (train) / R² (CV)',
                     'Regression', axes[1], color_val=REG_COLOR)
fig.suptitle('CV score plateaus as n_estimators grows — diminishing returns past ~100 trees',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# --- Sweep 2: max_features at n_estimators=100 ---
max_feat_grid = ['sqrt', 'log2', 0.3, 0.5, None]

clf_mf_means, clf_mf_sds = [], []
reg_mf_means, reg_mf_sds = [], []
for mf in max_feat_grid:
    m_c = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=RANDOM_SEED, n_jobs=-1)
    s_c = cross_val_score(m_c, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='roc_auc', n_jobs=-1)
    clf_mf_means.append(s_c.mean()); clf_mf_sds.append(s_c.std(ddof=1))

    m_r = RandomForestRegressor(n_estimators=100, max_features=mf, random_state=RANDOM_SEED, n_jobs=-1)
    s_r = cross_val_score(m_r, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    reg_mf_means.append(s_r.mean()); reg_mf_sds.append(s_r.std(ddof=1))

labels = [str(v) for v in max_feat_grid]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.bar(labels, clf_mf_means, yerr=clf_mf_sds, capsize=8, color=CLF_COLOR, edgecolor='black')
for i, m in enumerate(clf_mf_means):
    ax.text(i, m + 0.003, f'{m:.4f}', ha='center', fontsize=10)
ax.set_xlabel('max_features'); ax.set_ylabel('3-fold CV ROC-AUC')
ax.set_title('Classification — max_features sweep', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(labels, reg_mf_means, yerr=reg_mf_sds, capsize=8, color=REG_COLOR, edgecolor='black')
for i, m in enumerate(reg_mf_means):
    ax.text(i, m + 0.003, f'{m:.4f}', ha='center', fontsize=10)
ax.set_xlabel('max_features'); ax.set_ylabel('3-fold CV R²')
ax.set_title('Regression — max_features sweep', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('max_features controls tree diversity — smaller subsets often beat using all features',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The `n_estimators` curves on both spines tell the same story: a sharp lift from 10 → 50 trees, then a slow plateau past 100. By 200 trees the marginal improvement is in the third decimal place. For the headline plots and the M3 milestone, **`n_estimators=100` is the right working default** — beyond that you spend fit time without buying CV lift.

The `max_features` sweep is more interesting because the right answer is dataset-dependent. On classification, `sqrt(n_features)` (the sklearn default) usually wins or ties. On regression, the sklearn default of `None` (all features) often loses to `sqrt` or `0.3` — using all features at every split makes the trees too similar, which kills the diversity that makes bagging work. Tuning `max_features` to a fraction smaller than 1.0 typically buys 1–2 R² points on California housing.

A question that often comes up here is *"why is the regression default `None` if it's usually wrong?"* Historical reasons. The original Breiman paper proposed `sqrt(p)` for classification and `p/3` for regression; the sklearn `1.0` default is more permissive and matches a different tradition. The point is not to memorize the default but to **always tune `max_features` for regression projects** — a 30-second hyperparameter sweep typically produces a meaningful lift.

**Key takeaway:** `n_estimators=100` is a defensible default; `max_features` always deserves a sweep, especially on the regression spine.

---

## 6. Out-of-Bag (OOB) Score — Free Validation

Because each bootstrap leaves ~37% of training samples out, the random forest can score itself on those held-out samples **for free** — no extra train/test split needed. This is the OOB score: the prediction for sample `i` uses only the trees in which sample `i` was not in the bootstrap.

OOB and CV scores usually agree to within a fraction of a point. When they diverge meaningfully, something is wrong — typically because the data has dependence structure (groups, time) that bootstrap sampling does not respect. For our two datasets neither structure exists, so OOB and CV should track each other closely.

The plot below traces both signals as `n_estimators` grows. The OOB curve is a single value per `n_estimators` (no fold averaging needed); the CV curve is the 5-fold mean ± SD. They should converge as the forest matures.

In [ ]:
# OOB vs CV — paired
n_est_grid_oob = [25, 50, 100, 200, 300]

clf_oob, clf_cv_mean, clf_cv_std = [], [], []
for n in n_est_grid_oob:
    m = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(X_train_clf, y_train_clf)
    clf_oob.append(m.oob_score_)
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='accuracy', n_jobs=-1)
    clf_cv_mean.append(s.mean()); clf_cv_std.append(s.std(ddof=1))

reg_oob, reg_cv_mean, reg_cv_std = [], [], []
for n in n_est_grid_oob:
    m = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(X_train_reg, y_train_reg)
    reg_oob.append(m.oob_score_)  # OOB R²
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    reg_cv_mean.append(s.mean()); reg_cv_std.append(s.std(ddof=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
ax.plot(n_est_grid_oob, clf_oob, marker='o', linewidth=2, color=CLF_COLOR, label='OOB accuracy')
ax.errorbar(n_est_grid_oob, clf_cv_mean, yerr=clf_cv_std, marker='s', linewidth=2,
            capsize=5, color=GREY, label='3-fold CV accuracy ± SD')
ax.set_xlabel('n_estimators'); ax.set_ylabel('Accuracy')
ax.set_title('Classification — OOB vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(n_est_grid_oob, reg_oob, marker='o', linewidth=2, color=REG_COLOR, label='OOB R²')
ax.errorbar(n_est_grid_oob, reg_cv_mean, yerr=reg_cv_std, marker='s', linewidth=2,
            capsize=5, color=GREY, label='3-fold CV R² ± SD')
ax.set_xlabel('n_estimators'); ax.set_ylabel('R²')
ax.set_title('Regression — OOB vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('OOB tracks CV closely — a free validation signal at zero compute cost',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 OOB and CV curves should overlap. If they diverge, suspect data dependence (groups, time).")


**Reading the output:**

On both spines OOB and CV agree closely — they often differ by less than one SD. That agreement is the hallmark of i.i.d. data: when bootstrap sampling is appropriate, OOB and CV are estimating the same quantity. You get the OOB number for free during the forest fit; CV requires `k` separate fits at `k×` the cost.

When does OOB lie? When samples are not independent. Three common cases:
- **Time series:** future samples share a generative process with past ones — bootstrap mixes time periods, OOB optimistically high.
- **Hierarchical / grouped data** (e.g., multiple visits per patient): leaving out a row but keeping other rows from the same patient inflates OOB.
- **Panel data with strong fixed effects:** same as grouped, just a different name.

For the M3 milestone, OOB is a useful sanity check on top of CV — when both report the same number, you have two estimates with two different sets of assumptions agreeing, which is stronger evidence than either alone.

A question that often comes up here is *"if OOB is free and CV costs 5× more, why ever run CV?"* Three reasons. First, the **statistical machinery** (Student's *t* CIs, CI-overlap test) is built around CV folds — OOB gives one number, not a distribution. Second, CV with explicit folds lets you compare different model classes on identical splits, which OOB cannot (only forests have OOB; LogReg does not). Third, CV catches data-dependence problems by failing visibly — OOB silently agrees with itself even when both are wrong.

**Key takeaway:** OOB is the right second-opinion on CV when both apply; CV is the foundation for cross-model comparison. nb14 will use CV exclusively because the comparison is across model families, not within forests.

---

## 7. Feature Importance — The Four-Method Reconciliation Table

This is the section nb15's interpretation work leans on. There are at least four ways to ask *"which features matter most?"* and they often disagree. The disagreement is the pedagogical payload — when methods agree, the answer is unambiguous; when they disagree, you have to think about what each method is actually measuring.

The four methods we will reconcile:

| # | Method | What it measures | Cost |
|---|---|---|---|
| 1 | **Linear coefficient magnitude** (Week-2 reference) | Standardized β — change in target per SD change in feature, *holding others constant* | Free (one model fit) |
| 2 | **Impurity-based MDI** (`feature_importances_`) | Average decrease in node impurity weighted by samples reaching that node, summed across all trees | Free (one forest fit) |
| 3 | **Permutation importance** | Score drop when feature is randomly shuffled in evaluation | Moderate (`n_repeats × n_features` re-evaluations) |
| 4 | **Drop-column importance** | Score drop when feature is removed and the forest is **refit** without it | Expensive (`n_features` re-fits) |

The output is a **rank heatmap**: rows are features, columns are methods, cell values are ranks (1 = most important). When all four columns agree, the row is uniformly green; when methods disagree, the row is mottled. Both kinds of rows tell you something.

> 💡 **Gemini Prompt:** "Compute four importance methods on the breast cancer training set: standardized linear coefficient magnitudes from reference_clf; MDI from a fitted RandomForestClassifier(200 trees); permutation importance with 10 repeats; drop-column importance via 5-fold CV ROC-AUC for each feature dropped. Build a rank DataFrame (rows = features, columns = methods, values = rank). Render as a heatmap with the plot_importance_heatmap helper. Same four methods on California housing using R² and OLS."
>
> **After running, verify:**
> - [ ] Both spines produce a rank DataFrame with 4 columns (one per method)
> - [ ] Heatmap shows top-15 features per spine sorted by linear-coef rank
> - [ ] At least one feature has wildly different ranks across methods (the disagreement signal)


In [ ]:
# Helper: compute four-method importance ranks for a single spine
def four_method_ranks(reference_pipeline, ref_step_name,
                      forest_estimator,
                      X_train, y_train, scoring, cv, label,
                      drop_col_subset=None):
    """Returns a DataFrame with rows=features, columns=4 methods, values=rank."""
    feat_names = list(X_train.columns)

    # Method 1: linear coefficient magnitude (standardized)
    ref = reference_pipeline.fit(X_train, y_train)
    coef = ref.named_steps[ref_step_name].coef_
    coef = coef.ravel() if coef.ndim > 1 else coef
    coef_mag = np.abs(coef)

    # Method 2: MDI (impurity-based)
    forest = forest_estimator.fit(X_train, y_train)
    mdi = forest.feature_importances_

    # Method 3: permutation importance
    perm = permutation_importance(forest, X_train, y_train,
                                  scoring=scoring, n_repeats=10,
                                  random_state=RANDOM_SEED, n_jobs=-1)
    perm_mean = perm.importances_mean

    # Method 4: drop-column importance — refit without each feature, measure CV score drop
    base_score = cross_val_score(forest_estimator, X_train, y_train,
                                 cv=cv, scoring=scoring, n_jobs=-1).mean()
    drop_imp = []
    cols_to_drop = drop_col_subset if drop_col_subset is not None else feat_names
    drop_set = set(cols_to_drop)
    for col in feat_names:
        if col not in drop_set:
            drop_imp.append(np.nan)
            continue
        X_drop = X_train.drop(columns=[col])
        s = cross_val_score(forest_estimator, X_drop, y_train,
                            cv=cv, scoring=scoring, n_jobs=-1).mean()
        drop_imp.append(base_score - s)
    drop_imp = np.array(drop_imp, dtype=float)

    # Convert each method's importance to RANK (1 = most important).
    # NaN drop-importances (skipped features) get the worst rank.
    def to_rank(imp):
        s = pd.Series(imp, index=feat_names)
        return s.rank(ascending=False, method='min', na_option='bottom').astype(int)

    rank_df = pd.DataFrame({
        'Linear coef': to_rank(coef_mag),
        'MDI':         to_rank(mdi),
        'Permutation': to_rank(perm_mean),
        'Drop-column': to_rank(drop_imp),
    }, index=feat_names)
    print(f"=== {label} — top-10 features by linear coef rank ===")
    print(rank_df.sort_values('Linear coef').head(10).to_string())
    return rank_df, perm

# --- Classification: subset drop-column to the 10 features with highest MDI to keep runtime modest ---
quick_forest_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
mdi_quick = quick_forest_clf.fit(X_train_clf, y_train_clf).feature_importances_
top_clf_features = list(X_train_clf.columns[np.argsort(mdi_quick)[::-1][:10]])

ranks_clf, perm_clf = four_method_ranks(
    reference_pipeline=Pipeline([('scaler', StandardScaler()),
                                 ('clf', LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))]),
    ref_step_name='clf',
    forest_estimator=RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    X_train=X_train_clf, y_train=y_train_clf,
    scoring='roc_auc', cv=cv_clf_3, label='CLASSIFICATION',
    drop_col_subset=top_clf_features
)

# --- Regression: subset drop-column to top 5 by MDI to keep runtime under control ---
quick_forest_reg = RandomForestRegressor(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1)
mdi_quick_r = quick_forest_reg.fit(X_train_reg, y_train_reg).feature_importances_
top_reg_features = list(X_train_reg.columns[np.argsort(mdi_quick_r)[::-1][:5]])

ranks_reg, perm_reg = four_method_ranks(
    reference_pipeline=Pipeline([('scaler', StandardScaler()),
                                 ('reg', LinearRegression())]),
    ref_step_name='reg',
    forest_estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1),
    X_train=X_train_reg, y_train=y_train_reg,
    scoring='r2', cv=cv_reg_3, label='REGRESSION',
    drop_col_subset=top_reg_features
)


In [ ]:
# Render the four-method rank heatmaps
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
plot_importance_heatmap(ranks_clf, axes[0],
                        title='Classification (Wisconsin Breast Cancer) — feature rank across 4 methods',
                        top_n=15)
plot_importance_heatmap(ranks_reg, axes[1],
                        title='Regression (California Housing) — feature rank across 4 methods',
                        top_n=8)
plt.tight_layout()
plt.show()

print("\n💡 Rows that are uniformly green (low ranks across all 4 methods) are unambiguous winners.")
print("💡 Rows that are mottled (varying ranks) are where methods disagree — investigate why.")


**Reading the output:**

On classification, the top features in MDI / permutation / drop-column usually agree closely — `worst perimeter`, `worst concave points`, `worst radius`, `mean concave points` and a few related cell-shape measurements dominate every method. The linear-coefficient rank often differs because logistic regression spreads predictive power across correlated features (the "worst" and "mean" versions of the same measurement steal from each other), while the forest concentrates importance on whichever variant happened to be picked first in the trees. **That kind of disagreement is correlation talking** — not contradiction.

On regression, the top features (`MedInc`, `AveOccup`, `Latitude`, `Longitude`) usually agree across methods. `MedInc` (median income) is the single overwhelming predictor; the geography features matter but trade ranks across methods because of the same correlation effect. The disagreement to watch for is when MDI and permutation **disagree** on a feature with low cardinality — MDI inflates the importance of high-cardinality features (more split candidates), while permutation does not. If you see a categorical-like feature ranked high by MDI but low by permutation, the MDI rank is the artifact.

A question that often comes up here is *"if all four methods can disagree, which one do I trust?"* For the M3 milestone the safest default is the **permutation rank**, with MDI as a sanity check. Permutation directly measures *score drop when the feature is unavailable* — that is the closest analogue to the question you actually want answered (*"how much does this feature matter to predictions?"*) and it has no high-cardinality bias. Use the linear-coef rank as the **interpretability report**: tell the stakeholder *"controlling for the other features, a one-SD change in `MedInc` moves the predicted price by USD 53K"*. Use the drop-column rank only when permutation is suspect (e.g., highly correlated feature pairs where shuffling one is meaningless because the other still encodes it).

**Key takeaway:** Four methods, four perspectives. Agreement is a strong signal; disagreement is information about the data structure. nb15's interpretation pass uses this same table — bring this notebook's output forward.

---

## 8. Permutation Importance — Detail and Plotting

Permutation importance was already computed inside the four-method helper. This section visualizes it on its own with full error bars (10 repeats produces a SD per feature) — the format you will use directly on the M3 milestone poster.

**Why permutation importance specifically:**
- Model-agnostic — works on any fitted estimator, not just trees.
- Directly answers *"how much does the prediction quality fall if this feature is unavailable?"* — the question stakeholders actually want answered.
- Comes with built-in uncertainty (SD across repeats).

In [ ]:
# Permutation importance bar charts with error bars — one per spine
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
plot_importance_bars(perm_clf.importances_mean, X_train_clf.columns, axes[0],
                     errors=perm_clf.importances_std, color=CLF_COLOR,
                     title='Classification — permutation importance ± SD (10 repeats)', top_n=15)
plot_importance_bars(perm_reg.importances_mean, X_train_reg.columns, axes[1],
                     errors=perm_reg.importances_std, color=REG_COLOR,
                     title='Regression — permutation importance ± SD (10 repeats)', top_n=8)
fig.suptitle('Permutation importance — model-agnostic, with explicit uncertainty per feature',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The error bars are the part most readers underuse. A feature whose importance bar is short and whose error bar is wide enough to overlap zero is **not reliably important** — repeat the permutation enough times and it sometimes looks important and sometimes doesn't. Report the top features by mean **only when the SD is small enough that the rank is stable** under repeats.

For poster-ready figures, the rule of thumb is: include a feature in the importance discussion if its mean is at least 2× its SD. Below that, the importance estimate is too noisy to defend.

A question that often comes up here is *"why does permutation importance sometimes report negative values?"* Because the random shuffle can occasionally improve the score on a noisy feature — pure variance. Negative permutation importance is a synonym for *"this feature carries no signal; the random labels are no worse than the real ones."* Treat negative entries as zeros.

**Key takeaway:** Permutation importance is the headline plot for nb15 and your M3 poster. Always include the error bars; never report ranks for features whose mean falls below 2× SD.

---

## 9. Comprehensive Model Comparison — Tree vs Forest vs Week-2 Reference

The closing section of the notebook puts it all together: single tree (from nb11), random forest (today's contribution), and the Week-2 reference baseline (LogReg / OLS) on the same CV-CI dot plot per spine. This is the visual that nb14's selection ceremony will extend to a five-candidate field.

In [ ]:
# Comprehensive comparison — three models per spine
single_tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
forest_clf_v   = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
single_tree_reg = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_SEED)
forest_reg_v   = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)

clf_compare = {
    'Week-2 reference (LogReg)':       cross_val_score(reference_clf,   X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Single Tree (depth=3, nb11)':     cross_val_score(single_tree_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (100 trees)':       cross_val_score(forest_clf_v,    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_compare = {
    'Week-2 reference (OLS)':          cross_val_score(reference_reg,   X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Single Tree (depth=10, nb11)':    cross_val_score(single_tree_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Random Forest (100 trees)':       cross_val_score(forest_reg_v,    X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
}

print("=== CLASSIFICATION ===")
for name, s in clf_compare.items():
    print(f"  {name}:  mean = {s.mean():.4f}  SD = {s.std(ddof=1):.4f}")
print()
print("=== REGRESSION ===")
for name, s in reg_compare.items():
    print(f"  {name}:  mean = {s.mean():.4f}  SD = {s.std(ddof=1):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_cv_ci(clf_compare, 'ROC-AUC', 'Classification — comparison vs Week-2 reference', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_compare, 'R²',      'Regression — comparison vs Week-2 reference',     axes[1], color=REG_COLOR)
fig.suptitle('Forest beats both the single tree and (on regression) the Week-2 reference by a CI-clear margin',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On the regression spine the random forest's CV R² (typically ~0.80) clears the OLS reference (~0.60) by a CI-clear margin — well over 10 R² points of lift, with a CI that does not overlap. The forest has earned the right to displace the linear baseline.

On the classification spine the random forest's CV ROC-AUC is competitive with LogReg(C=1.0) but the **CIs overlap** — about 0.99 vs 0.998. The forest has not earned displacement on this dataset by the CI-overlap rule. This is exactly the kind of result nb14's selection ceremony will adjudicate formally; the answer here is that the linear baseline is hard to beat on a small, mostly-linearly-separable dataset.

A question that often comes up here is *"if the forest doesn't beat LogReg on classification, why use it?"* Three reasons. First, the forest gives you the four-method importance table you just produced — LogReg gives you only coefficients, which on highly correlated features (all the cell-shape measurements) are unreliable indicators of mechanism. Second, the forest is robust to misspecified preprocessing — if a future analyst fits LogReg without standardizing, the model breaks; the forest does not care about scale. Third, and most importantly: **CI-overlap means statistical tie, which means simpler model wins**. On this dataset, that is LogReg. The forest is a defensible alternative, not a strict improvement.

**Key takeaway:** The forest decisively beats OLS on regression and ties LogReg on classification. nb13 will introduce gradient boosting, which often pushes both spines higher; nb14 will adjudicate the full five-candidate field with the CI-clear-margin discipline.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification Forest

**Task:** Find the best `(n_estimators, max_features)` combination for `RandomForestClassifier` on Wisconsin breast cancer.

**Instructions:**
1. Sweep a 3×3 grid: `n_estimators ∈ [50, 100, 200]` × `max_features ∈ ['sqrt', 0.3, None]`.
2. For each combination, compute 5-fold CV ROC-AUC mean and SD using `cv_clf`.
3. Render as a heatmap with mean values annotated in each cell.
4. Pick the simplest combination whose CV mean is within one SD of the best (one-SE-rule).
5. Write 3 short findings: which dial moved the score most? Did the result confirm the `sqrt` default?

---

> 💡 **Gemini Prompt:** "Grid-search RandomForestClassifier(random_state=474, n_jobs=-1) over n_estimators=[50,100,200] × max_features=['sqrt',0.3,None] using 5-fold CV ROC-AUC on X_train_clf. Build a 3×3 heatmap of CV means with cell annotations and apply the one-SE rule to pick the simplest competitive combination."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated
> - [ ] Best combination and one-SE-rule pick both reported
> - [ ] All cells use n_jobs=-1


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune RandomForestClassifier over (n_estimators, max_features) using 5-fold CV ROC-AUC.
# Apply the one-SE rule.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression Forest

**Task:** Find the best `(n_estimators, max_features)` combination for `RandomForestRegressor` on California Housing.

**Instructions:**
1. Sweep a 3×3 grid: `n_estimators ∈ [50, 100, 200]` × `max_features ∈ ['sqrt', 0.3, None]`.
2. For each combination, compute 3-fold CV R² mean and SD using `cv_reg_3` (3-fold for runtime).
3. Render as a heatmap with mean values annotated in each cell.
4. Pick the simplest combination whose CV mean is within one SD of the best (one-SE-rule).
5. Write 3 short findings: did the regression case prefer a different `max_features` than classification? Why?

---

> 💡 **Gemini Prompt:** "Grid-search RandomForestRegressor(random_state=474, n_jobs=-1) over n_estimators=[50,100,200] × max_features=['sqrt',0.3,None] using 3-fold CV R² on X_train_reg. Build a 3×3 heatmap of CV means with cell annotations and apply the one-SE rule. Report the best CV-RMSE in USD."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated
> - [ ] Best combination and one-SE-rule pick both reported
> - [ ] Best CV-RMSE in USD printed


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune RandomForestRegressor over (n_estimators, max_features) using 3-fold CV R².
# Convert best CV-RMSE to USD; apply the one-SE rule.


## 10. Wrap-Up — Key Takeaways

**What landed today:**

1. **Bagging + random feature subsets reduce variance.** A forest of 100 trees has a tighter CV CI than a single tree on both spines — the variance-reduction-by-averaging math, made visible.
2. **OOB is a free second opinion on CV.** When OOB and CV agree, you have two estimates with different assumptions corroborating each other. When they disagree, suspect data dependence (groups, time).
3. **Four importance methods, four perspectives.** Linear-coef (interpretation), MDI (cheap), permutation (model-agnostic with uncertainty), drop-column (gold standard, expensive). Agreement is a strong signal; disagreement is information about the data.
4. **Forest beats OLS on regression by a CI-clear margin; ties LogReg on classification.** The Week-2 reference floor matters — and the forest only earns displacement on the regression spine.

**Bridge to nb13 — Gradient Boosting:**

Random forests reduce variance; gradient boosting reduces **bias**. Where forests fit many trees in parallel and average them, boosting fits trees sequentially — each one focused on the residuals (regression) or misclassifications (classification) of the previous ones. The result is often a CV-score lift of several points over the random forest, especially on regression with structure that a single deeper tree can capture but a shallow forest dilutes.

The same dual-spine pattern continues: `GradientBoostingClassifier` on Wisconsin breast cancer, `GradientBoostingRegressor` on California Housing, paired diagnostics at every step, both compared against the Week-2 reference floor. Bring today's tuning muscle memory — boosting needs `learning_rate`, `n_estimators`, and `max_depth` tuned together rather than independently.

A question that often comes up at this point is *"does gradient boosting always beat random forest?"* On most tabular datasets, yes — by a small margin. But the trade is real: boosting is sequential (slower to fit, harder to parallelize), more sensitive to hyperparameters (a wrong learning rate can wreck the model), and more prone to overfitting if you don't early-stop. nb13 walks both the wins and the trade-offs.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (RF tuning, classification) and Exercise 2 (RF tuning, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 12 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a tuning heatmap with the chosen combination starred
- [ ] The four-method importance heatmap renders for both spines
- [ ] All figures render (none broken)
- [ ] Both `_clf` and `_reg` variable namespaces stay disjoint (no `NameError`)

### Next Step:

- **Notebook 13** — Gradient Boosting (Day 13)

---

<center>

**Thank you!**

</center>